# 盈利修复方案：历史依据

## tl;dr
最新 13 轮中，10 轮首次修复报价若全成交即锁定毛亏损；15 秒内配对的 5 轮有 4 轮亏损。开仓成交观察到确认平均约 6.16 秒。单纯缩短 TTL 不足以修复收益目标。

## Context & Methods
只读分析 2026-10-17 的日志和已复核样本；最新 12 场、13 轮，加此前 36 场的执行统计作为机制对照。完整设计在 ../../RDP_REBALANCE_PROFIT_REPAIR_PLAN_20261018.md。

### Key Assumptions
报价收益是若按开仓数量全成交的算术条件，不是成交模拟；开仓快照不代表后来成交时仍有相同价格或数量。官方撮合时间及完整前两秒订单簿/排队历史缺失。新方案参数仅为候选，不是回测证明的盈利参数。

## Data
代码复验 12 份原始 journal 哈希，并读取两个旧批次的复核结果。无交易 API 调用。

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/polymaker').is_dir())
REVIEW = ROOT / 'docs/development/reviews/20261018_repair_design'
"""Historical observations supporting a profit-aware repair proposal, not a backtest."""
import csv
import hashlib
import json
import statistics
from decimal import Decimal
from pathlib import Path

OUT=REVIEW
BASE=OUT.parent/'20261017_2245_2345'
D=lambda x:Decimal(str(x))


def main():
    data=json.loads((BASE/'analysis.json').read_text())
    paired=json.loads((OUT.parent/'20261017_2245_2345_pairing/evidence.json').read_text())
    observations=[]
    sources=[]
    for m in data['markets']:
        p=Path(m['source'])
        raw=p.read_bytes()
        digest=hashlib.sha256(raw).hexdigest()
        assert digest==next(s['sha256'] for s in data['provenance'] if s['path']==m['source'])
        sources.append(dict(path=str(p),sha256=digest))
        rows=[json.loads(l) for l in raw.decode().splitlines()]
        intents={r['data']['payload']['intent']['window_id']:r['data']['payload']['intent']
                 for r in rows if r['kind']=='sm_obi_final_decision'}
        windows=[w for w in data['windows'] if w['session_id']==m['session_id']]
        for c in [c for c in paired['cycles'] if c['session_id']==m['session_id']]:
            w=next(w for w in windows if w['window']==c['window'])
            start=windows.index(w)
            end=next((i for i in range(start+1,len(windows))
                      if windows[i]['phase']=='NORMAL' and windows[i]['fill_quantity']),len(windows))
            repairs=[w for w in windows[start+1:end] if w['phase']=='REBALANCE' and w['submitted']]
            first=repairs[0]
            intent=intents[w['session_id']+':'+w['window']]
            snapshot=intent.get('final_snapshot') or intent['snapshot']
            other=snapshot['no' if w['side']=='YES' else 'yes']
            price,qty=D(w['price']),D(w['fill_quantity'])
            bid,ask=D(other['best_bid']),D(other['best_ask'])
            first_price=D(first['price'])
            normalfills=[f for f in data['fills'] if f['order_id']==w['order_id']]
            lag=min(f['matched_observed_epoch'] for f in normalfills)-w['post_epoch']
            observations.append(dict(market=m['market'],window=w['window'],side=w['side'],
                entry_price=float(price),quantity=float(qty),entry_obi=c['entry_signal'],
                opposite_bid_at_entry=float(bid),opposite_ask_at_entry=float(ask),
                entry_plus_opposite_bid=float(price+bid),entry_plus_opposite_ask=float(price+ask),
                same_quantity_bid_repair_meets_one_dollar_minimum=bid*qty>=1,
                first_repair_window=first['window'],first_repair_price=float(first_price),
                first_repair_outcome=first['outcome_final'],
                first_repair_hypothetical_gross=float(qty*(1-price-first_price)),
                final_fifo_pair_gross=c['fifo_acquisition_pair_gross'],
                pair_seconds=c['matched_to_matched_s'],repair_submissions=c['repair_submissions'],
                entry_post_to_first_matched_observation_s=lag,
                first_repair_actual_filled_quantity=first['fill_quantity'],
                source=m['source'],entry_decision_line=w['source_line'],first_repair_decision_line=first['source_line']))
    old=[]
    for name in ['20261017_1300_1500','20261017_1540_1640']:
        p=OUT.parent/name/'analysis.json'
        a=json.loads(p.read_text())
        old.append(dict(cohort=name,markets=a['totals']['markets'],
            rebalance_posts=a['totals']['phases']['REBALANCE'],
            stale_closes=a['totals']['phase_close_reasons']['REBALANCE']['WS_BOOK_STALE'],
            residual_markets=a['totals']['residual_markets']))
        sources.append(dict(path=str(p.resolve()),sha256=hashlib.sha256(p.read_bytes()).hexdigest()))
    fast=[o for o in observations if o['pair_seconds']<=15]
    result=dict(scope='Latest 12 markets and preceding 36 reviewed markets, 2026-10-17; observed historical outcomes only',
        latest=dict(cycles=len(observations),first_repair_quotes_lock_negative_gross=sum(o['first_repair_hypothetical_gross']<0 for o in observations),
            final_positive_cycles=sum(o['final_fifo_pair_gross']>0 for o in observations),
            final_negative_cycles=sum(o['final_fifo_pair_gross']<0 for o in observations),
            fast_within_15_s=dict(cycles=len(fast),negative_cycles=sum(o['final_fifo_pair_gross']<0 for o in fast),
                fifo_pair_gross_sum=sum(o['final_fifo_pair_gross'] for o in fast)),
            entry_plus_opposite_bid_levels=sorted(set(o['entry_plus_opposite_bid'] for o in observations)),
            entry_plus_opposite_ask_levels=sorted(set(o['entry_plus_opposite_ask'] for o in observations)),
            entry_snapshots_with_exact_quantity_bid_repair_below_minimum=sum(not o['same_quantity_bid_repair_meets_one_dollar_minimum'] for o in observations),
            opening_post_to_matched_observation_mean_s=statistics.mean(o['entry_post_to_first_matched_observation_s'] for o in observations),
            opening_matched_observation_at_least_2_s_after_post=sum(o['entry_post_to_first_matched_observation_s']>=2 for o in observations),
            opening_matched_to_confirmed_mean_s=paired['all']['metrics']['entry_confirmation_wait_s']['mean'],
            reversed_filled_normal_orders=paired['counts']['signal_reversed_closes'],
            reversed_close_post_latency_median_s=paired['reversed_normal_post_to_close_s']['median'],
            rebalance_posts=data['totals']['phases']['REBALANCE'],
            stale_closes=data['totals']['phase_close_reasons']['REBALANCE']['WS_BOOK_STALE']),
        previous=old,observations=observations,sources=sources,
        caveats=['First repair quoted PnL is arithmetic under full hypothetical fill, not an actual outcome.',
            'Entry snapshot bid/ask sums do not prove depth, queue priority or fill availability when the opening later matched.',
            'FIFO pair gross excludes fees/rebates and differs slightly from average-cost merge accounting for dust.',
            'No complete event-by-event executable order-book and queue replay covers the first two seconds after every fill.',
            'A two-second horizon and proposed target/stop levels are design hypotheses, not fitted profitable parameters.'])
    assert result['latest']['first_repair_quotes_lock_negative_gross']==10
    assert result['latest']['entry_plus_opposite_bid_levels']==[.99]
    assert result['latest']['entry_plus_opposite_ask_levels']==[1.0]
    (OUT/'evidence.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
    with (OUT/'observations.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(observations[0]),lineterminator='\n')
        writer.writeheader();writer.writerows(observations)
    print(json.dumps(result['latest'],ensure_ascii=False,indent=2))
    return result


if __name__=='__main__':
    RESULT=main()


{
  "cycles": 13,
  "first_repair_quotes_lock_negative_gross": 10,
  "final_positive_cycles": 4,
  "final_negative_cycles": 9,
  "fast_within_15_s": {
    "cycles": 5,
    "negative_cycles": 4,
    "fifo_pair_gross_sum": -2.6
  },
  "entry_plus_opposite_bid_levels": [
    0.99
  ],
  "entry_plus_opposite_ask_levels": [
    1.0
  ],
  "entry_snapshots_with_exact_quantity_bid_repair_below_minimum": 1,
  "opening_post_to_matched_observation_mean_s": 2.3228259820204515,
  "opening_matched_observation_at_least_2_s_after_post": 8,
  "opening_matched_to_confirmed_mean_s": 6.163560959009024,
  "reversed_filled_normal_orders": 11,
  "reversed_close_post_latency_median_s": 0.864828834310174,
  "rebalance_posts": 80,
  "stale_closes": 55
}


## Results

逐轮区分首次报价算术和最终实际成交结果。以下全部毛收益，不是最终净收益。

In [2]:
for o in RESULT['observations']:
    print(f"{o['market']} {o['window']:>3} entry={o['entry_price']:.2f} first_repair={o['first_repair_price']:.2f} quote_gross={o['first_repair_hypothetical_gross']:+.2f} actual_FIFO_gross={o['final_fifo_pair_gross']:+.4f} paired={o['pair_seconds']:.2f}s")

22:45  w7 entry=0.66 first_repair=0.40 quote_gross=-0.30 actual_FIFO_gross=+0.6000 paired=51.19s
23:00 w16 entry=0.24 first_repair=0.80 quote_gross=-0.20 actual_FIFO_gross=-0.9500 paired=74.13s
23:05 w11 entry=0.54 first_repair=0.47 quote_gross=-0.05 actual_FIFO_gross=-0.8476 paired=53.59s
23:10  w1 entry=0.42 first_repair=0.54 quote_gross=+0.20 actual_FIFO_gross=+0.0500 paired=16.15s
23:15  w1 entry=0.51 first_repair=0.44 quote_gross=+0.25 actual_FIFO_gross=+1.0500 paired=26.81s
23:20  w8 entry=0.91 first_repair=0.36 quote_gross=-1.35 actual_FIFO_gross=-1.3500 paired=13.00s
23:20 w13 entry=0.32 first_repair=0.69 quote_gross=-0.05 actual_FIFO_gross=-0.0500 paired=11.56s
23:25  w1 entry=0.52 first_repair=0.63 quote_gross=-0.75 actual_FIFO_gross=-0.7500 paired=10.21s
23:25  w3 entry=0.53 first_repair=0.57 quote_gross=-0.50 actual_FIFO_gross=-0.5000 paired=7.28s
23:25  w7 entry=0.69 first_repair=0.90 quote_gross=-2.95 actual_FIFO_gross=-2.9500 paired=180.37s
23:30  w1 entry=0.29 first_rep

## Takeaways

改动必须同时包含收益限价、成交事件触发、原信号总时间预算及超时风险管理。maker-only 无法保证立即成交；若允许修复主动成交，必须另外计算费用和合法数量。13 次开仓快照中对侧 Ask 与入场价之和均为 1，不能假设立即买对侧有正利润。

本 Notebook 只验证建议依据，没有回测新策略，不计算反事实盈利。完整机制、参数候选和验证步骤见方案文档。